# AWS FinOps by Linked Account — 2026 actuals → 2027 forecast
One Cost Explorer pull from the **roll-up account** *Entercom Communications* (`724972922289`), grouped by
**LINKED_ACCOUNT × SERVICE** (every member account), plus **SERVICE × BILLING_ENTITY** to split AWS Services from Marketplace.
The 2027 forecast uses **only 2026 data** — Services and Marketplace separately, and by account.

Logic lives in `AWS_FinOps_By_Account.py` (terminal: `python AWS_FinOps_By_Account.py`, or `--cached` to skip AWS).

In [ ]:
import importlib, re
import pandas as pd
import AWS_FinOps_By_Account as A
import finops_exec_report as fr
importlib.reload(A)
pd.set_option('display.float_format', '{:,.0f}'.format)

## Parameters

In [ ]:
USE_CACHE = False                 # True = reuse AWS_2026_By_Account.csv (no AWS login)
INCLUDE_CURRENT_MONTH = False     # month-to-date is pulled but excluded from the forecast basis either way
MONTHLY_BUDGET = 500_000          # AWS Cloud Services budget per month (Marketplace excluded)

## Pull 2026 from Cost Explorer (approve the SSO sign-in in your browser)

In [ ]:
df = A.load_cached() if USE_CACHE else A.pull(INCLUDE_CURRENT_MONTH)
print(f"{len(df):,} rows · {df['AccountId'].nunique()} linked accounts · {df['Date'].min():%b %Y} – {df['Date'].max():%b %Y}")
df.pivot_table(index='Date', columns='Category', values='Cost', aggfunc='sum')

## Data check — totals vs the Monthly_Cost_For_*.csv extracts

In [ ]:
A.reconcile_with_monthly_files(df)

## 2026 by account

In [ ]:
(df[df['Category'] == A.SVC].pivot_table(index='AccountName', columns='Date', values='Cost', aggfunc='sum')
   .fillna(0).assign(Total=lambda t: t.sum(axis=1)).sort_values('Total', ascending=False)
   .rename(columns=lambda c: c.strftime('%b') if hasattr(c, 'strftime') else c))

# 2027 forecast (2026 basis only)

In [ ]:
fc = A.forecast_2027(df, MONTHLY_BUDGET)
for title, lines in fc.summary['paragraphs']:
    print(f'\n{title.upper()}')
    for line in lines:
        print('  •', re.sub('<[^>]+>', '', line))

## a) Cloud Services

In [ ]:
A.chart_services(fc).show()
display(A.quarterly_table(fc))
display(A.monthly_table(fc))

## Cloud Services by linked account

In [ ]:
A.chart_accounts_share(fc).show()
display(fc.accounts)
from types import SimpleNamespace
A.fd.chart_small_multiples(SimpleNamespace(lines={'acct': fc.account_lines}), 'acct', A.C_SVC).show()

In [ ]:
for acct, t in fc.account_services.items():
    print(acct)
    display(t)

## b) Marketplace

In [ ]:
A.chart_marketplace(fc).show()
display(fc.contracts)
display(fc.anomalies)   # account × service lines that dropped to ~$0 (credits, adjustments, shutdowns)

## c) Combined

In [ ]:
A.chart_combined(fc).show()

## Write the HTML report

In [ ]:
print('Report written:', A.build_report(fc, recon=A.reconcile_with_monthly_files(df)))